# Python OOP — Veri Bilimi ve ML Üzerinden

Fonksiyonlar ve sınıfları soyut örneklerle değil, gerçek bir veri bilimcinin yazdığı kodla öğreniyoruz.

---
## İçindekiler
1. [Fonksiyonlar — Veri Temizleme Araçları](#1-fonksiyonlar--veri-temizleme-araçları)
2. [*args / **kwargs — Esnek Metrik Fonksiyonları](#2-args--kwargs--esnek-metrik-fonksiyonları)
3. [Lambda & Higher-Order Functions — Özellik Mühendisliği](#3-lambda--higher-order-functions--özellik-mühendisliği)
4. [Sınıfa Giriş — DataLoader](#4-sınıfa-giriş--dataloader)
5. [Kalıtım — Model Sınıf Hiyerarşisi](#5-kalıtım--model-sınıf-hiyerarşisi)
6. [Dunder Metotlar — DataFrame Wrapper](#6-dunder-metotlar--dataframe-wrapper)
7. [Property & Encapsulation — ModelRegistry](#7-property--encapsulation--modelregistry)
8. [Soyut Sınıf — Transformers Pipeline](#8-soyut-sınıf--transformers-pipeline)
9. [Tam Proje — MLPipeline Sınıfı](#9-tam-proje--mlpipeline-sınıfı)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from abc import ABC, abstractmethod
from sklearn.datasets import load_iris, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams['figure.figsize'] = (10, 4)
sns.set_theme(style='whitegrid')
print("✅ Hazır.")

---
## 1. Fonksiyonlar — Veri Temizleme Araçları

Bir veri bilimci, aynı temizleme adımlarını defalarca tekrarlamak yerine fonksiyonlara paketler.

**Öğretilen kavramlar:** `def`, `return`, docstring, varsayılan parametre

In [ ]:
# ── Fonksiyon 1: Eksik değer raporu ──────────────────────────────────────────
def eksik_deger_raporu(df):
    """
    DataFrame'deki eksik değerlerin sayısını ve oranını döndürür.
    Eksik değer içermeyen sütunları filtreler.
    """
    eksik = df.isnull().sum()
    oran  = eksik / len(df) * 100
    rapor = pd.DataFrame({'Eksik Sayı': eksik, 'Oran (%)': oran.round(1)})
    return rapor[rapor['Eksik Sayı'] > 0].sort_values('Oran (%)', ascending=False)


# ── Fonksiyon 2: IQR ile aykırı değer kırpma ─────────────────────────────────
def iqr_kirp(df, sutun, carpan=1.5):
    """
    Belirtilen sütundaki aykırı değerleri IQR yöntemiyle kırpar.
    carpan: IQR çarpanı (varsayılan 1.5, katı için 3.0)
    """
    Q1 = df[sutun].quantile(0.25)
    Q3 = df[sutun].quantile(0.75)
    IQR = Q3 - Q1
    alt, ust = Q1 - carpan * IQR, Q3 + carpan * IQR
    onceki = len(df)
    temiz  = df[df[sutun].between(alt, ust)]
    print(f"'{sutun}': {onceki - len(temiz)} aykırı değer kaldırıldı  "
          f"({alt:.1f} – {ust:.1f})")
    return temiz


# ── Fonksiyon 3: Kategorik sütun özeti ───────────────────────────────────────
def kategorik_ozet(df, sutun, en_fazla=5):
    """
    Kategorik bir sütunun frekans dağılımını gösterir.
    en_fazla: gösterilecek maksimum kategori sayısı
    """
    sayim = df[sutun].value_counts(dropna=False)
    oran  = sayim / len(df) * 100
    tablo = pd.DataFrame({'Sayı': sayim, 'Oran (%)': oran.round(1)}).head(en_fazla)
    print(f"── {sutun} ({df[sutun].nunique()} benzersiz) ──")
    print(tablo.to_string())
    return tablo


# ─── Demo ────────────────────────────────────────────────────────────────────
n = 300
demo = pd.DataFrame({
    'yas':    np.random.normal(35, 10, n),
    'maas':   np.random.normal(6000, 2000, n),
    'sehir':  np.random.choice(['İstanbul','Ankara','İzmir','Bursa'], n),
})
# Kasıtlı eksik + aykırı değer
demo.loc[np.random.choice(n,30,replace=False), 'yas']  = np.nan
demo.loc[np.random.choice(n,20,replace=False), 'maas'] = np.nan
demo.loc[np.random.choice(n, 5,replace=False), 'maas'] = 99999

print("=== Eksik Değer Raporu ===")
display(eksik_deger_raporu(demo))

print("\n=== Aykırı Değer Kırpma ===")
temiz = iqr_kirp(demo.dropna(), 'maas')

print("\n=== Kategorik Özet ===")
kategorik_ozet(demo, 'sehir')

---
## 2. *args / **kwargs — Esnek Metrik Fonksiyonları

Bir model, birden fazla metrikle değerlendirilebilir. `**kwargs` ile hangi metriklerin hesaplanacağını çalışma zamanında seçebiliriz.

**Öğretilen kavramlar:** `*args`, `**kwargs`, fonksiyon nesnesi olarak argüman

In [ ]:
# *args → birden fazla model tahmini karşılaştır
def tahmin_karşilaştir(y_gercek, *tahminler, isimler=None):
    """
    Birden fazla model tahminini doğruluk açısından karşılaştırır.
    *tahminler: sınırsız sayıda tahmin dizisi
    """
    isimler = isimler or [f'Model_{i+1}' for i in range(len(tahminler))]
    sonuclar = []
    for isim, t in zip(isimler, tahminler):
        acc = accuracy_score(y_gercek, t)
        f1  = f1_score(y_gercek, t, average='weighted')
        sonuclar.append({'Model': isim, 'Accuracy': acc, 'F1': f1})
    return pd.DataFrame(sonuclar).set_index('Model').round(4)


# **kwargs → hangi metrikleri hesaplayacağını dışarıdan geç
def degerlendirici(y_gercek, y_tahmin, **metrikler):
    """
    İstenilen sklearn metriklerini anahtar-değer çifti olarak kabul eder.
    Kullanım: degerlendirici(y, yp, dogruluk=accuracy_score, f1=f1_score)
    """
    sonuclar = {}
    for ad, fonk in metrikler.items():
        try:
            # Bazı metrikler ek parametre ister
            sonuc = fonk(y_gercek, y_tahmin)
        except TypeError:
            sonuc = fonk(y_gercek, y_tahmin, average='weighted')
        sonuclar[ad] = round(sonuc, 4)
        print(f"  {ad:<15}: {sonuc:.4f}")
    return sonuclar


# ─── Demo ────────────────────────────────────────────────────────────────────
iris = load_iris()
X, y = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

lr   = LogisticRegression(max_iter=200).fit(X_tr, y_tr)
dt   = DecisionTreeClassifier(max_depth=3).fit(X_tr, y_tr)
rf   = RandomForestClassifier(n_estimators=50).fit(X_tr, y_tr)

t_lr, t_dt, t_rf = lr.predict(X_te), dt.predict(X_te), rf.predict(X_te)

print("=== *args: Üç Modeli Karşılaştır ===")
display(tahmin_karşilaştir(y_te, t_lr, t_dt, t_rf,
                           isimler=['Loj. Reg.', 'Karar Ağacı', 'Rnd. Forest']))

print("\n=== **kwargs: Seçilen Metrikler ===")
degerlendirici(y_te, t_rf,
               dogruluk=accuracy_score,
               f1_skoru=f1_score)

---
## 3. Lambda & Higher-Order Functions — Özellik Mühendisliği

Veri biliminde `map`, `filter`, `apply` çok sık kullanılır. Bunların hepsi fonksiyonu argüman olarak alır.

**Öğretilen kavramlar:** `lambda`, `map`, `filter`, `sorted`, `DataFrame.apply`

In [ ]:
df = pd.DataFrame({
    'yas':     [23, 35, 17, 45, 29, 52, 14, 38],
    'gelir':   [2800, 7500, 1200, 9500, 4200, 12000, 800, 6800],
    'cinsiyet':['E', 'K', 'E', 'K', 'E', 'K', 'E', 'K'],
})

# ── Lambda ile yeni özellik türetme ──────────────────────────────────────────
df['yas_grubu'] = df['yas'].apply(lambda x:
    'Genç'   if x < 25 else
    'Orta'   if x < 45 else
    'Kıdemli'
)

df['gelir_log'] = df['gelir'].apply(lambda x: np.log1p(x))

# ── filter: yetişkin ve geliri ortalamanın üstündekileri filtrele ─────────────
ortalama_gelir = df['gelir'].mean()
secim = list(filter(
    lambda i: df.loc[i, 'yas'] >= 18 and df.loc[i, 'gelir'] > ortalama_gelir,
    df.index
))
print(f"Yetişkin ve geliri >{ortalama_gelir:.0f} TL olan satır sayısı: {len(secim)}")

# ── sorted: modelleri CV skoruna göre sırala ──────────────────────────────────
modeller = {
    'Loj. Reg.':      lr,
    'Karar Ağacı':    dt,
    'Rnd. Forest':    rf,
}
cv_skorlar = {
    isim: cross_val_score(model, X, y, cv=5).mean()
    for isim, model in modeller.items()
}
sirali = sorted(cv_skorlar.items(), key=lambda item: item[1], reverse=True)

print("\n── Modeller CV Skoruna Göre (yüksekten düşüğe) ──")
for sira, (isim, skor) in enumerate(sirali, 1):
    print(f"  {sira}. {isim:<18} → {skor:.4f}")

# ── map ile sütun kodlama ─────────────────────────────────────────────────────
df['cinsiyet_kod'] = df['cinsiyet'].map({'E': 0, 'K': 1})

print("\n── Özellik Mühendisliği Sonucu ──")
display(df)

---
## 4. Sınıfa Giriş — DataLoader

Bir veri bilimci sıkça aynı veri yükleme + ön işleme adımlarını tekrarlar.  
Bunları bir sınıfta paketleyince durumu (state) da saklayabiliriz.

**Öğretilen kavramlar:** `class`, `__init__`, `self`, örnek değişkeni, örnek metodu

In [ ]:
class DataLoader:
    """Bir veri setini yükler, temizler ve bölümlere ayırır."""

    # Sınıf değişkeni — tüm DataLoader nesnelerinin paylaştığı sayaç
    yuklenen_dataset_sayisi = 0

    def __init__(self, df: pd.DataFrame, hedef_sutun: str, test_oran: float = 0.2):
        self.df            = df.copy()
        self.hedef_sutun   = hedef_sutun
        self.test_oran     = test_oran
        self.X_train       = None
        self.X_test        = None
        self.y_train       = None
        self.y_test        = None
        self._hazir        = False        # private durum bayrağı
        DataLoader.yuklenen_dataset_sayisi += 1
        print(f"📂 DataLoader oluşturuldu — {df.shape[0]} satır, {df.shape[1]} sütun")

    # ── Metot 1: temel istatistikler ─────────────────────────────────────────
    def ozet(self):
        """Veri setinin kısa özetini yazdırır."""
        print(f"Boyut      : {self.df.shape}")
        print(f"Hedef      : '{self.hedef_sutun}'")
        print(f"Eksik(toplam): {self.df.isnull().sum().sum()}")
        print(f"Hedef dağılımı:\n{self.df[self.hedef_sutun].value_counts().to_string()}")

    # ── Metot 2: temizle ─────────────────────────────────────────────────────
    def temizle(self, strateji='median'):
        """Sayısal eksik değerleri doldurur, kategorikleri modla doldurur."""
        for sutun in self.df.columns:
            if self.df[sutun].isnull().sum() == 0:
                continue
            if self.df[sutun].dtype in ['float64', 'int64']:
                deger = self.df[sutun].median() if strateji == 'median' else self.df[sutun].mean()
                self.df[sutun].fillna(deger, inplace=True)
            else:
                self.df[sutun].fillna(self.df[sutun].mode()[0], inplace=True)
        print(f"✅ Temizleme tamam — kalan eksik: {self.df.isnull().sum().sum()}")
        return self   # method chaining için

    # ── Metot 3: böl ─────────────────────────────────────────────────────────
    def bol(self):
        """Eğitim / test setlerine ayırır."""
        X = self.df.drop(columns=[self.hedef_sutun])
        y = self.df[self.hedef_sutun]
        self.X_train, self.X_test, self.y_train, self.y_test = \
            train_test_split(X, y, test_size=self.test_oran, random_state=42, stratify=y)
        self._hazir = True
        print(f"✂️  Eğitim: {len(self.X_train)}  |  Test: {len(self.X_test)}")
        return self

    def hazir_mi(self):
        return self._hazir


# ─── Demo ────────────────────────────────────────────────────────────────────
iris_df = load_iris(as_frame=True).frame
iris_df.columns = [*load_iris().feature_names, 'hedef']
# Kasıtlı eksik değer
iris_df.loc[[5,10,20], 'sepal length (cm)'] = np.nan

# Method chaining
yukleme = DataLoader(iris_df, hedef_sutun='hedef', test_oran=0.2)
yukleme.ozet()
print()
yukleme.temizle().bol()   # zincirleme

print(f"\nHazır mı? {yukleme.hazir_mi()}")
print(f"Toplam yüklenen dataset sayısı: {DataLoader.yuklenen_dataset_sayisi}")

---
## 5. Kalıtım — Model Sınıf Hiyerarşisi

Her ML modeli eğitilir, tahmin yapar ve değerlendirilir.  
Ortak davranışları üst sınıfa, model-özel olanları alt sınıflara koyarız.

**Öğretilen kavramlar:** kalıtım, `super()`, metot ezme (override), `isinstance`

In [ ]:
# ── Üst sınıf ────────────────────────────────────────────────────────────────
class TemelModel:
    """Tüm modellerin ortak arayüzü."""

    def __init__(self, model_adi, versiyon='1.0'):
        self.model_adi  = model_adi
        self.versiyon   = versiyon
        self._egitildi  = False
        self.skor       = None

    def egit(self, X_train, y_train):
        raise NotImplementedError("Alt sınıf bu metodu uygulamalı.")

    def degerlendir(self, X_test, y_test):
        """Doğruluk ve F1 hesaplar — tüm alt sınıflar miras alır."""
        if not self._egitildi:
            print("⚠️  Model henüz eğitilmedi!")
            return
        tahmin  = self.tahmin_et(X_test)
        acc     = accuracy_score(y_test, tahmin)
        f1      = f1_score(y_test, tahmin, average='weighted')
        self.skor = acc
        print(f"[{self.model_adi}]  Accuracy: {acc:.4f}  |  F1: {f1:.4f}")
        return acc, f1

    def tahmin_et(self, X):
        raise NotImplementedError

    def __str__(self):
        durum = f"skor={self.skor:.4f}" if self.skor else "eğitilmedi"
        return f"{self.model_adi} v{self.versiyon} [{durum}]"


# ── Alt sınıf 1 ──────────────────────────────────────────────────────────────
class LojistikModel(TemelModel):
    def __init__(self, C=1.0):
        super().__init__('LojistikRegresyon', versiyon='1.0')
        self.C     = C
        self._sk   = LogisticRegression(C=C, max_iter=500, random_state=42)

    def egit(self, X_train, y_train):
        self._sk.fit(X_train, y_train)
        self._egitildi = True
        print(f"✅ {self.model_adi} eğitildi  (C={self.C})")
        return self

    def tahmin_et(self, X):
        return self._sk.predict(X)


# ── Alt sınıf 2 ──────────────────────────────────────────────────────────────
class OrmanlModel(TemelModel):
    def __init__(self, n_agac=100, derinlik=None):
        super().__init__('RandomForest', versiyon='2.0')
        self.n_agac  = n_agac
        self._sk     = RandomForestClassifier(
            n_estimators=n_agac, max_depth=derinlik, random_state=42)

    def egit(self, X_train, y_train):
        self._sk.fit(X_train, y_train)
        self._egitildi = True
        print(f"✅ {self.model_adi} eğitildi  (n_agac={self.n_agac})")
        return self

    def tahmin_et(self, X):
        return self._sk.predict(X)

    # Alt sınıfa özgü ek metot
    def ozellik_onemleri(self, ozellik_isimleri):
        """Yalnızca Random Forest'a özgü özellik önemi grafiği."""
        onem = pd.Series(self._sk.feature_importances_, index=ozellik_isimleri)
        onem.sort_values().plot(kind='barh', color='#3498db', edgecolor='white')
        plt.title('Özellik Önemi')
        plt.tight_layout()
        plt.show()


# ─── Demo ────────────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(yukleme.X_train)
X_te_s = scaler.transform(yukleme.X_test)

loj = LojistikModel(C=0.5).egit(X_tr_s, yukleme.y_train)
rf  = OrmanlModel(n_agac=100).egit(yukleme.X_train, yukleme.y_train)

print()
loj.degerlendir(X_te_s, yukleme.y_test)
rf.degerlendir(yukleme.X_test, yukleme.y_test)

print(f"\n{loj}")
print(f"{rf}")

print(f"\nloj bir TemelModel mi? {isinstance(loj, TemelModel)}")
print(f"loj bir OrmanlModel mi? {isinstance(loj, OrmanlModel)}")

print()
rf.ozellik_onemleri(load_iris().feature_names)

---
## 6. Dunder Metotlar — DataFrame Wrapper

Bir `Dataset` sınıfının `len`, `[]`, `+` gibi Python operatörlerine doğal yanıt vermesini sağlarız.

**Öğretilen kavramlar:** `__len__`, `__getitem__`, `__add__`, `__repr__`, `__contains__`

In [ ]:
class Dataset:
    """Makine öğrenmesi için hafif DataFrame sarmalayıcı."""

    def __init__(self, df: pd.DataFrame, ad: str = 'Dataset'):
        self._df = df.copy()
        self.ad  = ad

    # len(dataset)
    def __len__(self):
        return len(self._df)

    # dataset[0]  veya  dataset['sutun']
    def __getitem__(self, anahtar):
        if isinstance(anahtar, int):
            return self._df.iloc[anahtar]
        return self._df[anahtar]

    # dataset1 + dataset2  → satırları birleştir
    def __add__(self, diger):
        birlesik = pd.concat([self._df, diger._df], ignore_index=True)
        return Dataset(birlesik, ad=f"{self.ad}+{diger.ad}")

    # 'sutun_adi' in dataset
    def __contains__(self, sutun):
        return sutun in self._df.columns

    # repr(dataset)
    def __repr__(self):
        return (f"Dataset(ad='{self.ad}', "
                f"satir={len(self._df)}, "
                f"sutun={list(self._df.columns)})")

    # for satir in dataset
    def __iter__(self):
        return self._df.iterrows()

    def sutunlar(self):
        return list(self._df.columns)

    def goster(self, n=3):
        display(self._df.head(n))


# ─── Demo ────────────────────────────────────────────────────────────────────
ds1 = Dataset(iris_df.iloc[:80], ad='Egitim')
ds2 = Dataset(iris_df.iloc[80:], ad='Test')

print(f"len(ds1)          : {len(ds1)}")
print(f"'hedef' in ds1    : {'hedef' in ds1}")
print(f"'yas' in ds1      : {'yas' in ds1}")
print(f"\nds1[0] (1. satır):")
print(ds1[0])

birlesik = ds1 + ds2
print(f"\nBirleşik: {repr(birlesik)}")

---
## 7. Property & Encapsulation — ModelRegistry

Eğitilen modelleri saklamak, karşılaştırmak ve en iyisine erişmek için bir kayıt (registry) sınıfı.

**Öğretilen kavramlar:** `@property`, `setter`, private değişken, kapsülleme

In [ ]:
class ModelRegistry:
    """Eğitilmiş modelleri kaydeder, karşılaştırır ve en iyisini döndürür."""

    def __init__(self):
        self.__modeller = {}         # private: doğrudan erişim yok
        self.__aktif_model = None    # private

    # ── @property: __modeller'e salt-okunur kopy ─────────────────────────────
    @property
    def modeller(self):
        """Kayıtlı modellerin salt-okunur kopyası."""
        return dict(self.__modeller)

    @property
    def en_iyi(self):
        """En yüksek skorlu modeli döndürür."""
        if not self.__modeller:
            return None
        return max(self.__modeller.values(), key=lambda m: m.skor or 0)

    @property
    def aktif_model(self):
        return self.__aktif_model

    @aktif_model.setter
    def aktif_model(self, model_adi):
        """Aktif modeli isimle ayarla — kayıtlı olmalı."""
        if model_adi not in self.__modeller:
            raise KeyError(f"'{model_adi}' kayıtlı değil. Önce kaydet.")
        self.__aktif_model = self.__modeller[model_adi]
        print(f"🔵 Aktif model: {model_adi}")

    # ── Metotlar ─────────────────────────────────────────────────────────────
    def kaydet(self, model: TemelModel):
        """Skor hesaplanmış bir modeli kaydet."""
        if model.skor is None:
            print(f"⚠️  {model.model_adi} değerlendirilmedi, önce degerlendir() çalıştır.")
            return
        self.__modeller[model.model_adi] = model
        print(f"💾 Kaydedildi: {model.model_adi}  (skor={model.skor:.4f})")

    def karsilastir(self):
        """Tüm kayıtlı modelleri tablo olarak göster."""
        if not self.__modeller:
            print("Kayıtlı model yok.")
            return
        tablo = pd.DataFrame(
            [(m.model_adi, m.versiyon, f"{m.skor:.4f}")
             for m in self.__modeller.values()],
            columns=['Model', 'Versiyon', 'Skor']
        ).set_index('Model')
        display(tablo)

    def tahmin_et(self, X):
        """Aktif model ile tahmin yap."""
        if self.__aktif_model is None:
            raise RuntimeError("Aktif model ayarlanmadı.")
        return self.__aktif_model.tahmin_et(X)


# ─── Demo ────────────────────────────────────────────────────────────────────
registry = ModelRegistry()

registry.kaydet(loj)
registry.kaydet(rf)

print("\n── Karşılaştırma Tablosu ──")
registry.karsilastir()

print(f"\nEn iyi model: {registry.en_iyi}")

# Property setter
registry.aktif_model = 'RandomForest'
tahminler = registry.tahmin_et(yukleme.X_test)
print(f"İlk 5 tahmin: {tahminler[:5]}")

# Doğrudan erişim engellenir
try:
    registry.__modeller
except AttributeError as e:
    print(f"\nPrivate erişim engellendi: {e}")

---
## 8. Soyut Sınıf — Transformers Pipeline

sklearn'ün `Transformer` mantığı aslında soyut sınıf prensibiyle çalışır.  
Her dönüşüm adımı `fit`, `transform`, `fit_transform` sunmak **zorundadır**.

**Öğretilen kavramlar:** `ABC`, `@abstractmethod`, polimorfizm

In [ ]:
# ── Soyut temel dönüştürücü ───────────────────────────────────────────────────
class TemelDonusturucu(ABC):
    """Her veri dönüştürücünün uygulaması gereken arayüz."""

    def __init__(self, ad):
        self.ad        = ad
        self._uyarlandi = False

    @abstractmethod
    def uyarla(self, X):
        """Eğitim verisinden istatistik öğren."""
        pass

    @abstractmethod
    def donustur(self, X):
        """Öğrenilen istatistikle veriyi dönüştür."""
        pass

    def uyarla_donustur(self, X):
        """Kısayol: uyarla + dönüştür."""
        return self.uyarla(X).donustur(X)

    def __repr__(self):
        return f"{self.ad}(uyarlandı={self._uyarlandi})"


# ── Somut sınıf 1: Z-score normalize ─────────────────────────────────────────
class ZScoreNormalizer(TemelDonusturucu):
    def __init__(self):
        super().__init__('ZScoreNormalizer')
        self.ortalama_ = None
        self.std_       = None

    def uyarla(self, X):
        self.ortalama_ = np.mean(X, axis=0)
        self.std_       = np.std(X, axis=0) + 1e-8   # sıfır std önlemi
        self._uyarlandi = True
        print(f"✅ {self.ad} uyarlandı")
        return self

    def donustur(self, X):
        if not self._uyarlandi:
            raise RuntimeError("Önce uyarla() çalıştırın.")
        return (X - self.ortalama_) / self.std_


# ── Somut sınıf 2: Min-Max normalize ─────────────────────────────────────────
class MinMaxNormalizer(TemelDonusturucu):
    def __init__(self, aralik=(0, 1)):
        super().__init__('MinMaxNormalizer')
        self.aralik = aralik
        self.min_   = None
        self.max_   = None

    def uyarla(self, X):
        self.min_ = np.min(X, axis=0)
        self.max_ = np.max(X, axis=0)
        self._uyarlandi = True
        print(f"✅ {self.ad} uyarlandı")
        return self

    def donustur(self, X):
        if not self._uyarlandi:
            raise RuntimeError("Önce uyarla() çalıştırın.")
        a, b  = self.aralik
        aralik = self.max_ - self.min_ + 1e-8
        return a + (X - self.min_) / aralik * (b - a)


# ── Polimorfizm: her iki dönüştürücü aynı arayüzle kullanılıyor ──────────────
X_raw = np.array(yukleme.X_train)
X_te_raw = np.array(yukleme.X_test)

for donusturucu in [ZScoreNormalizer(), MinMaxNormalizer()]:
    X_d = donusturucu.uyarla(X_raw).donustur(X_te_raw)
    print(f"  {donusturucu.ad}:  min={X_d.min():.3f}  max={X_d.max():.3f}  "
          f"ortalama={X_d.mean():.3f}")

# Doğrudan örnekleme dener misin?
try:
    t = TemelDonusturucu('deneme')
except TypeError as e:
    print(f"\nSoyut sınıf örneklenemez: {e}")

---
## 9. Tam Proje — MLPipeline Sınıfı

Tüm adımları tek bir `MLPipeline` sınıfında birleştiriyoruz:  
yükleme → ön işleme → eğitim → değerlendirme → görselleştirme.

In [ ]:
class MLPipeline:
    """
    Veri yükleme, ön işleme, çoklu model eğitimi ve
    karşılaştırmalı değerlendirmeyi tek sınıfta yönetir.
    """

    def __init__(self, veri: pd.DataFrame, hedef: str):
        self.__veri     = veri.copy()
        self.__hedef    = hedef
        self.__modeller = {}       # isim → sklearn modeli
        self.__sonuclar = {}       # isim → (acc, f1)
        self._X_train = self._X_test = self._y_train = self._y_test = None
        self._scaler   = StandardScaler()

    # ── Adım 1: hazırla ──────────────────────────────────────────────────────
    def hazirla(self, test_oran=0.2):
        df = self.__veri.dropna()
        X  = df.drop(columns=[self.__hedef])
        y  = df[self.__hedef]
        self._X_train, self._X_test, self._y_train, self._y_test = \
            train_test_split(X, y, test_size=test_oran,
                             random_state=42, stratify=y)
        self._X_train = self._scaler.fit_transform(self._X_train)
        self._X_test  = self._scaler.transform(self._X_test)
        print(f"📊 Hazırlandı  → Eğitim:{len(self._y_train)}  Test:{len(self._y_test)}")
        return self

    # ── Adım 2: model ekle ───────────────────────────────────────────────────
    def model_ekle(self, **modeller):
        """Anahtar kelime argümanlarıyla modelleri kaydet.
        Örnek: pipeline.model_ekle(lr=LogisticRegression(), rf=RandomForestClassifier())
        """
        self.__modeller.update(modeller)
        print(f"➕ Eklendi: {list(modeller.keys())}")
        return self

    # ── Adım 3: eğit ve değerlendir ──────────────────────────────────────────
    def calistir(self):
        if self._X_train is None:
            raise RuntimeError("Önce hazirla() çalıştırın.")
        for isim, model in self.__modeller.items():
            model.fit(self._X_train, self._y_train)
            tahmin = model.predict(self._X_test)
            acc    = accuracy_score(self._y_test, tahmin)
            f1     = f1_score(self._y_test, tahmin, average='weighted')
            self.__sonuclar[isim] = {'Accuracy': acc, 'F1': f1}
        print("\n🏁 Eğitim tamamlandı.")
        return self

    # ── Adım 4: karşılaştır ──────────────────────────────────────────────────
    @property
    def sonuclar(self):
        if not self.__sonuclar:
            return None
        return pd.DataFrame(self.__sonuclar).T.round(4)

    @property
    def en_iyi(self):
        if not self.__sonuclar:
            return None
        return max(self.__sonuclar, key=lambda k: self.__sonuclar[k]['Accuracy'])

    # ── Adım 5: görselleştir ─────────────────────────────────────────────────
    def gorsellestir(self):
        if not self.__sonuclar:
            print("Önce calistir() çalıştırın.")
            return

        tablo = self.sonuclar
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))

        for ax, metrik in zip(axes, ['Accuracy', 'F1']):
            renkler = ['#e74c3c' if m == self.en_iyi else '#3498db'
                       for m in tablo.index]
            bars = ax.bar(tablo.index, tablo[metrik], color=renkler,
                          edgecolor='white', zorder=3)
            ax.set_ylim(tablo[metrik].min() * 0.95, 1.01)
            ax.set_title(metrik)
            ax.set_ylabel('Skor')
            ax.grid(axis='y', zorder=0)
            for bar, val in zip(bars, tablo[metrik]):
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.002,
                        f'{val:.4f}', ha='center', fontsize=10)

        plt.suptitle(f'Model Karşılaştırması  |  En iyi: {self.en_iyi}',
                     fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

    def __repr__(self):
        return (f"MLPipeline(hedef='{self.__hedef}', "
                f"model_sayisi={len(self.__modeller)})")


# ─── Tek akışlı kullanım ─────────────────────────────────────────────────────
pipeline = (
    MLPipeline(iris_df, hedef='hedef')
    .hazirla(test_oran=0.2)
    .model_ekle(
        Lojistik   = LogisticRegression(max_iter=500, C=1.0),
        KararAgaci = DecisionTreeClassifier(max_depth=5),
        RndForest  = RandomForestClassifier(n_estimators=100)
    )
    .calistir()
)

display(pipeline.sonuclar)
print(f"\n🏆 En iyi model: {pipeline.en_iyi}")
pipeline.gorsellestir()

---
## Özet — Kavram ↔ ML Karşılığı

| OOP / Fonksiyon Kavramı | Veri Bilimi / ML Karşılığı |
|---|---|
| `def` fonksiyon | `eksik_deger_raporu`, `iqr_kirp` gibi yeniden kullanılabilir araçlar |
| Varsayılan parametre | `iqr_kirp(df, 'maas', carpan=1.5)` |
| `*args` | Sınırsız modeli birden karşılaştırmak |
| `**kwargs` | Hangi metriklerin hesaplanacağını dışarıdan geçmek |
| `lambda` | `df.apply`, `sorted`, `filter` içinde satır içi dönüşüm |
| `class` | `DataLoader`, `Dataset`, `MLPipeline` |
| `__init__` | Model hiperparametrelerini ve durumu başlatmak |
| Kalıtım | `LojistikModel`, `OrmanlModel` → `TemelModel`'den miras |
| `super()` | Ortak `model_adi`, `versiyon` başlatma |
| Dunder metotlar | `len(dataset)`, `dataset['sutun']`, `ds1 + ds2` |
| `@property` | `registry.en_iyi`, `pipeline.sonuclar` |
| Private (`__`) | `__modeller`, `__veri` → dışarıdan değiştirilemesin |
| `ABC` | `TemelDonusturucu` → alt sınıflar `uyarla` ve `donustur` uygulamalı |
| Polimorfizm | `ZScoreNormalizer` ve `MinMaxNormalizer` aynı arayüzle çalışır |